# Incident Response Runbook: Persistence - Boot or Logon Autostart Execution

**Tactic:** Persistence
**Technique:** Boot or Logon Autostart Execution (T1547)
**Severity:** HIGH

## Overview

This runbook provides step-by-step incident response procedures for Boot or Logon Autostart Execution activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import subprocess
import os

# Import security integrations
from autobook.runbook_loader import *
from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

alert_data = {
    'alert_id': 'ALERT-001',
    'timestamp': datetime.now().isoformat(),
    'tactic': 'Persistence',
    'technique': 'Boot or Logon Autostart Execution',
    'severity': 'HIGH',
    'incident_id': f"IR-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
}

print(f"Alert ID: {alert_data['alert_id']}")
print(f"Timestamp: {alert_data['timestamp']}")
print(f"Tactic: {alert_data['tactic']}")
print(f"Technique: {alert_data['technique']}")
print(f"Severity: {alert_data['severity']}")
print(f"Incident ID: {alert_data['incident_id']}")

# Query Splunk for boot/logon autostart execution indicators
print(f"\n[ACTION] Querying Splunk for boot/logon autostart execution indicators...")
autostart_queries = [
    # Registry Run keys modifications
    'index=* sourcetype=WinEventLog:Security EventCode=4657 "HKLM\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run" OR "HKCU\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run"',
    # Startup folder modifications
    'index=* sourcetype=WinEventLog:Security "Startup" "folder" "modification" OR "shell:startup"',
    # Service creation/modification
    'index=* sourcetype=WinEventLog:System EventCode=7036 "service" "created" OR "service" "modified"',
    # Scheduled task creation
    'index=* sourcetype=WinEventLog:Security EventCode=4698 OR EventCode=4702 "scheduled task" "created"',
    # WMI persistence
    'index=* sourcetype=WinEventLog:Security "WMI" "subscription" OR "WMI" "event" "consumer"'
]

suspicious_activities = []
affected_systems = []

for query in autostart_queries:
    try:
        results = splunk.search(query, time_range='-24h')
        if results:
            suspicious_activities.extend(results)
            print(f"   Found {len(results)} autostart persistence indicators")
        else:
            print(f"  - No results for query: {query[:50]}...")
    except Exception as e:
        print(f"   Query failed: {e}")

# Analyze with CrowdStrike for autostart persistence patterns
print(f"\n[ACTION] Analyzing with CrowdStrike for autostart persistence patterns...")
for activity in suspicious_activities:
    try:
        host_analysis = crowdstrike.analyze_host(activity.get('host', ''))
        if host_analysis.get('autostart_persistence_detected'):
            affected_systems.append({
                'hostname': activity.get('host', ''),
                'device_id': host_analysis.get('device_id', ''),
                'persistence_mechanisms': host_analysis.get('persistence_mechanisms', []),
                'autostart_entries': host_analysis.get('autostart_entries', [])
            })
            print(f"   Autostart persistence detected on {activity.get('host', '')}: {len(host_analysis.get('persistence_mechanisms', []))} mechanisms, {len(host_analysis.get('autostart_entries', []))} entries")
    except Exception as e:
        print(f"   Host analysis failed for {activity.get('host', '')}: {e}")

# Enrich with threat intelligence
print(f"\n[ACTION] Enriching with threat intelligence...")
for activity in suspicious_activities:
    try:
        # Search for autostart entries or persistence mechanisms in MISP
        if activity.get('registry_key'):
            misp_results = misp.search_iocs(activity['registry_key'], type='regkey')
            if misp_results:
                activity['threat_intel'] = misp_results[:3]
                print(f"   Threat intel found for registry key: {len(misp_results)} matches")
        elif activity.get('file_path'):
            misp_results = misp.search_iocs(activity['file_path'], type='filename')
            if misp_results:
                activity['threat_intel'] = misp_results[:3]
                print(f"   Threat intel found for file path: {len(misp_results)} matches")
    except Exception as e:
        print(f"   Threat intel lookup failed: {e}")

# Create IRIS case
print(f"\n[ACTION] Creating IRIS incident case...")
try:
    iris_case = iris.create_case(
        title=f"Persistence - Boot or Logon Autostart Execution Incident {alert_data['incident_id']}",
        description=f"Detected boot or logon autostart execution persistence on {len(affected_systems)} systems with {len(suspicious_activities)} suspicious indicators.",
        severity='high',
        tags=['persistence', 'autostart-execution', 't1547']
    )
    alert_data['iris_case_id'] = iris_case.get('case_id')
    print(f"   IRIS case created: {iris_case.get('case_id')}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")

print(f"\n Detection complete:")
print(f"  - {len(suspicious_activities)} suspicious autostart activities detected")
print(f"  - {len(affected_systems)} affected systems identified")
print(f"  - Threat intelligence enriched: {len([a for a in suspicious_activities if a.get('threat_intel')])} activities")
print(f"  - IRIS case created: {alert_data.get('iris_case_id', 'none')}")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

isolated_hosts = []
blocked_entities = []
disabled_autostarts = []

# Isolate affected systems immediately
print(f"\n[ACTION] Isolating affected systems...")
for system in affected_systems:
    try:
        isolation_result = crowdstrike.isolate_host(system['device_id'])
        if isolation_result.get('status') == 'isolated':
            isolated_hosts.append(system['hostname'])
            print(f"   Isolated {system['hostname']}")
        else:
            print(f"   Failed to isolate {system['hostname']}: {isolation_result.get('message', 'unknown error')}")
    except Exception as e:
        print(f"   Isolation failed for {system.get('hostname', 'unknown')}: {e}")

# Block associated IPs and domains
print(f"\n[ACTION] Blocking associated IPs and domains...")
for activity in suspicious_activities:
    try:
        if activity.get('src_ip'):
            block_result = shuffle.block_ip(activity['src_ip'])
            if block_result.get('status') == 'blocked':
                blocked_entities.append(f"IP:{activity['src_ip']}")
                print(f"   Blocked source IP: {activity['src_ip']}")

        if activity.get('domain'):
            domain_block = shuffle.block_domain(activity['domain'])
            if domain_block.get('status') == 'blocked':
                blocked_entities.append(f"Domain:{activity['domain']}")
                print(f"   Blocked domain: {activity['domain']}")
    except Exception as e:
        print(f"   Entity blocking failed: {e}")

# Disable suspicious autostart entries
print(f"\n[ACTION] Disabling suspicious autostart entries...")
for system in affected_systems:
    try:
        for entry in system.get('autostart_entries', []):
            disable_result = crowdstrike.disable_autostart_entry(
                system['device_id'],
                entry.get('type', ''),
                entry.get('key', '')
            )
            if disable_result.get('status') == 'disabled':
                disabled_autostarts.append(f"{system['hostname']}:{entry.get('key', '')}")
                print(f"   Disabled autostart entry: {entry.get('key', '')} on {system['hostname']}")
    except Exception as e:
        print(f"   Autostart disabling failed for {system.get('hostname', 'unknown')}: {e}")

# Terminate processes started by autostart mechanisms
print(f"\n[ACTION] Terminating processes started by autostart mechanisms...")
terminated_processes = []
for activity in suspicious_activities:
    try:
        if activity.get('process_id') and activity.get('host'):
            termination_result = crowdstrike.terminate_process(
                activity['host'],
                activity['process_id']
            )
            if termination_result.get('status') == 'terminated':
                terminated_processes.append(f"{activity['host']}:{activity['process_id']}")
                print(f"   Terminated autostart process {activity['process_id']} on {activity['host']}")
    except Exception as e:
        print(f"   Process termination failed: {e}")

# Quarantine autostart files
print(f"\n[ACTION] Quarantining autostart files...")
quarantined_files = []
for system in affected_systems:
    try:
        for entry in system.get('autostart_entries', []):
            if entry.get('file_path'):
                quarantine_result = crowdstrike.quarantine_file(
                    system['hostname'],
                    entry['file_path']
                )
                if quarantine_result.get('status') == 'quarantined':
                    quarantined_files.append(entry['file_path'])
                    print(f"   Quarantined autostart file: {entry['file_path']}")
    except Exception as e:
        print(f"   File quarantine failed: {e}")

# Enable enhanced monitoring for persistence mechanisms
print(f"\n[ACTION] Enabling enhanced monitoring for persistence mechanisms...")
try:
    monitoring_result = shuffle.enable_enhanced_monitoring(
        targets=[s['hostname'] for s in affected_systems],
        rules=['autostart_modification_detection', 'registry_persistence', 'scheduled_task_creation', 'service_persistence']
    )
    if monitoring_result.get('status') == 'enabled':
        print(f"   Enhanced persistence monitoring enabled for {len(affected_systems)} systems")
except Exception as e:
    print(f"   Enhanced monitoring setup failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_hosts)} systems isolated")
print(f"  - {len(blocked_entities)} entities blocked")
print(f"  - {len(disabled_autostarts)} autostart entries disabled")
print(f"  - {len(terminated_processes)} autostart processes terminated")
print(f"  - {len(quarantined_files)} autostart files quarantined")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

removed_entries = []
cleaned_registries = []
deleted_services = []

# Remove autostart entries
print(f"\n[ACTION] Removing autostart entries...")
for system in affected_systems:
    try:
        for entry in system.get('autostart_entries', []):
            removal_result = crowdstrike.remove_autostart_entry(
                system['device_id'],
                entry.get('type', ''),
                entry.get('key', '')
            )
            if removal_result.get('status') == 'removed':
                removed_entries.append(f"{system['hostname']}:{entry.get('key', '')}")
                print(f"   Removed autostart entry: {entry.get('key', '')} from {system['hostname']}")
    except Exception as e:
        print(f"   Autostart entry removal failed: {e}")

# Clean registry of persistence entries
print(f"\n[ACTION] Cleaning registry of persistence entries...")
for system in affected_systems:
    try:
        registry_clean = crowdstrike.clean_persistence_registry(system['device_id'])
        if registry_clean.get('status') == 'cleaned':
            cleaned_registries.append(system['hostname'])
            print(f"   Cleaned persistence registry entries on {system['hostname']}")
    except Exception as e:
        print(f"   Registry cleaning failed for {system.get('hostname', 'unknown')}: {e}")

# Remove scheduled tasks
print(f"\n[ACTION] Removing scheduled tasks...")
removed_tasks = []
for system in affected_systems:
    try:
        task_removal = crowdstrike.remove_scheduled_tasks(system['device_id'])
        if task_removal.get('status') == 'removed':
            removed_tasks.extend(task_removal.get('removed_tasks', []))
            print(f"   Removed {len(task_removal.get('removed_tasks', []))} scheduled tasks from {system['hostname']}")
    except Exception as e:
        print(f"   Scheduled task removal failed for {system.get('hostname', 'unknown')}: {e}")

# Delete malicious services
print(f"\n[ACTION] Deleting malicious services...")
for system in affected_systems:
    try:
        for mechanism in system.get('persistence_mechanisms', []):
            if mechanism.get('type') == 'service':
                service_deletion = crowdstrike.delete_service(
                    system['device_id'],
                    mechanism.get('service_name', '')
                )
                if service_deletion.get('status') == 'deleted':
                    deleted_services.append(f"{system['hostname']}:{mechanism.get('service_name', '')}")
                    print(f"   Deleted malicious service: {mechanism.get('service_name', '')} from {system['hostname']}")
    except Exception as e:
        print(f"   Service deletion failed: {e}")

# Remove WMI persistence
print(f"\n[ACTION] Removing WMI persistence...")
removed_wmi = []
for system in affected_systems:
    try:
        wmi_removal = crowdstrike.remove_wmi_persistence(system['device_id'])
        if wmi_removal.get('status') == 'removed':
            removed_wmi.append(system['hostname'])
            print(f"   Removed WMI persistence from {system['hostname']}")
    except Exception as e:
        print(f"   WMI persistence removal failed for {system.get('hostname', 'unknown')}: {e}")

# Clean startup folders
print(f"\n[ACTION] Cleaning startup folders...")
cleaned_folders = []
for system in affected_systems:
    try:
        folder_clean = crowdstrike.clean_startup_folders(system['device_id'])
        if folder_clean.get('status') == 'cleaned':
            cleaned_folders.append(system['hostname'])
            print(f"   Cleaned startup folders on {system['hostname']}")
    except Exception as e:
        print(f"   Startup folder cleaning failed for {system.get('hostname', 'unknown')}: {e}")

# Verify threat removal
print(f"\n[ACTION] Verifying threat removal...")
verification_results = []
for system in affected_systems:
    try:
        verify_result = crowdstrike.verify_persistence_removal(system['device_id'])
        verification_results.append({
            'system': system['hostname'],
            'status': 'clean' if verify_result.get('persistence_detected', True) == False else 'threats_remaining',
            'remaining_indicators': verify_result.get('remaining_indicators', 0)
        })
        status = "" if verify_result.get('persistence_detected', True) == False else ""
        print(f"  {status} Verification for {system['hostname']}: {verify_result.get('remaining_indicators', 0)} persistence indicators remaining")
    except Exception as e:
        print(f"   Verification failed for {system.get('hostname', 'unknown')}: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(removed_entries)} autostart entries removed")
print(f"  - {len(cleaned_registries)} systems had registry cleaned")
print(f"  - {len(removed_tasks)} scheduled tasks removed")
print(f"  - {len(deleted_services)} malicious services deleted")
print(f"  - {len(removed_wmi)} systems had WMI persistence removed")
print(f"  - {len(cleaned_folders)} systems had startup folders cleaned")
print(f"  - {len([v for v in verification_results if v['status'] == 'clean'])} systems verified clean")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

restored_systems = []
restarted_services = []
validated_functions = []

# Restore system functionality
print(f"\n[ACTION] Restoring system functionality...")
for system in affected_systems:
    try:
        restore_result = crowdstrike.restore_system_functionality(system['device_id'])
        if restore_result.get('status') == 'restored':
            restored_systems.append(system['hostname'])
            print(f"   Restored system functionality on {system['hostname']}")
    except Exception as e:
        print(f"   System restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Restart critical services
print(f"\n[ACTION] Restarting critical services...")
for system in affected_systems:
    try:
        service_restart = crowdstrike.restart_critical_services(system['device_id'])
        if service_restart.get('status') == 'restarted':
            restarted_services.append(system['hostname'])
            print(f"   Restarted critical services on {system['hostname']}")
    except Exception as e:
        print(f"   Service restart failed for {system.get('hostname', 'unknown')}: {e}")

# Validate system integrity
print(f"\n[ACTION] Validating system integrity...")
integrity_checks = []
for system in affected_systems:
    try:
        integrity_result = crowdstrike.validate_system_integrity(system['device_id'])
        integrity_checks.append({
            'system': system['hostname'],
            'integrity_status': integrity_result.get('integrity_status', 'unknown'),
            'issues_found': len(integrity_result.get('issues', []))
        })
        status = "" if integrity_result.get('integrity_status') == 'valid' else ""
        print(f"  {status} Integrity check for {system['hostname']}: {integrity_result.get('integrity_status', 'unknown')} ({len(integrity_result.get('issues', []))} issues)")
    except Exception as e:
        print(f"   Integrity validation failed for {system.get('hostname', 'unknown')}: {e}")

# Restore from backup if needed
print(f"\n[ACTION] Checking for backup restoration needs...")
backup_restores = []
for system in affected_systems:
    try:
        backup_check = crowdstrike.check_backup_restoration(system['device_id'])
        if backup_check.get('restoration_needed', False):
            restore_result = crowdstrike.restore_from_backup(system['device_id'])
            if restore_result.get('status') == 'restored':
                backup_restores.append(system['hostname'])
                print(f"   Restored {system['hostname']} from backup")
        else:
            print(f"   No backup restoration needed for {system['hostname']}")
    except Exception as e:
        print(f"   Backup restoration check failed for {system.get('hostname', 'unknown')}: {e}")

# Re-enable security controls
print(f"\n[ACTION] Re-enabling security controls...")
reenabled_controls = []
for system in affected_systems:
    try:
        controls_result = crowdstrike.reenable_security_controls(system['device_id'])
        if controls_result.get('status') == 'enabled':
            reenabled_controls.append(system['hostname'])
            print(f"   Re-enabled security controls on {system['hostname']}")
    except Exception as e:
        print(f"   Security controls re-enabling failed for {system.get('hostname', 'unknown')}: {e}")

# Perform functional testing
print(f"\n[ACTION] Performing functional testing...")
functional_tests = []
for system in affected_systems:
    try:
        test_result = crowdstrike.perform_functional_testing(system['device_id'])
        functional_tests.append({
            'system': system['hostname'],
            'tests_passed': test_result.get('tests_passed', 0),
            'total_tests': test_result.get('total_tests', 0)
        })
        pass_rate = test_result.get('tests_passed', 0) / max(test_result.get('total_tests', 1), 1) * 100
        status = "" if pass_rate >= 95 else ""
        print(f"  {status} Functional tests for {system['hostname']}: {test_result.get('tests_passed', 0)}/{test_result.get('total_tests', 0)} passed ({pass_rate:.1f}%)")
    except Exception as e:
        print(f"   Functional testing failed for {system.get('hostname', 'unknown')}: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} systems had functionality restored")
print(f"  - {len(restarted_services)} systems had critical services restarted")
print(f"  - {len([i for i in integrity_checks if i['integrity_status'] == 'valid'])} systems passed integrity checks")
print(f"  - {len(backup_restores)} systems restored from backup")
print(f"  - {len(reenabled_controls)} systems had security controls re-enabled")
print(f"  - {len([f for f in functional_tests if f['tests_passed'] == f['total_tests']])} systems passed all functional tests")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident
print("\n" + "=" * 60)
print("STEP 5: Post-Incident")
print("=" * 60)

# Update IRIS case with eradication results
print(f"\n[ACTION] Updating IRIS case with eradication results...")
try:
    eradication_summary = {
        'removed_entries': len(removed_entries),
        'cleaned_registries': len(cleaned_registries),
        'removed_tasks': len(removed_tasks),
        'deleted_services': len(deleted_services),
        'removed_wmi': len(removed_wmi),
        'cleaned_folders': len(cleaned_folders),
        'verification_results': verification_results
    }
    iris.update_case(incident_id, 'eradication_complete', eradication_summary)
    print(f"   Updated IRIS case {incident_id} with eradication results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

# Update IRIS case with recovery results
print(f"\n[ACTION] Updating IRIS case with recovery results...")
try:
    recovery_summary = {
        'restored_systems': len(restored_systems),
        'restarted_services': len(restarted_services),
        'integrity_checks': integrity_checks,
        'backup_restores': len(backup_restores),
        'reenabled_controls': len(reenabled_controls),
        'functional_tests': functional_tests
    }
    iris.update_case(incident_id, 'recovery_complete', recovery_summary)
    print(f"   Updated IRIS case {incident_id} with recovery results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

# Generate incident report
print(f"\n[ACTION] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'technique': 'Persistence: Boot or Logon Autostart Execution',
        'detection_time': detection_time,
        'affected_systems': len(affected_systems),
        'timeline': {
            'detection': detection_time,
            'containment': containment_time,
            'eradication': datetime.now().isoformat(),
            'recovery': datetime.now().isoformat()
        },
        'actions_taken': {
            'detection': f"Found {len(affected_systems)} systems with autostart persistence",
            'containment': f"Isolated {len(isolated_hosts)} hosts, disabled {len(disabled_entries)} autostart entries",
            'eradication': f"Removed {len(removed_entries)} entries, cleaned {len(cleaned_registries)} registries",
            'recovery': f"Restored {len(restored_systems)} systems, restarted {len(restarted_services)} services"
        },
        'indicators': splunk_indicators,
        'threat_intel': misp_results,
        'lessons_learned': [
            "Implement stricter autostart entry monitoring",
            "Regular registry and startup folder audits",
            "Enhanced EDR rules for persistence detection"
        ]
    }
    iris.generate_report(incident_id, incident_report)
    print(f"   Generated comprehensive incident report for case {incident_id}")
except Exception as e:
    print(f"   Report generation failed: {e}")

# Share IOCs with MISP
print(f"\n[ACTION] Sharing IOCs with MISP...")
try:
    misp_iocs = []
    for indicator in splunk_indicators:
        if indicator.get('type') in ['file', 'registry', 'process']:
            misp_iocs.append({
                'type': indicator['type'],
                'value': indicator['value'],
                'tags': ['autostart-persistence', 'incident-' + str(incident_id)]
            })

    if misp_iocs:
        misp.share_iocs(misp_iocs, f"Autostart Persistence Incident {incident_id}")
        print(f"   Shared {len(misp_iocs)} IOCs with MISP community")
    else:
        print(f"   No new IOCs to share with MISP")
except Exception as e:
    print(f"   MISP IOC sharing failed: {e}")

# Close IRIS case
print(f"\n[ACTION] Closing IRIS case...")
try:
    iris.close_case(incident_id, 'resolved')
    print(f"   Closed IRIS case {incident_id} as resolved")
except Exception as e:
    print(f"   IRIS case closure failed: {e}")

# Final status summary
print(f"\n Post-incident activities complete:")
print(f"  - IRIS case {incident_id} updated and closed")
print(f"  - Incident report generated")
print(f"  - {len(misp_iocs) if 'misp_iocs' in locals() else 0} IOCs shared with threat intelligence community")
print(f"  - All 5 IR phases completed successfully")

# Calculate incident metrics
incident_duration = (datetime.now() - datetime.fromisoformat(detection_time)).total_seconds() / 3600
print(f"\n📊 Incident Metrics:")
print(f"  - Duration: {incident_duration:.2f} hours")
print(f"  - Systems affected: {len(affected_systems)}")
print(f"  - Containment time: {'< 1 hour' if incident_duration < 1 else f'{incident_duration:.1f} hours'}")
print(f"  - Eradication success: {len([v for v in verification_results if v['status'] == 'clean'])}/{len(affected_systems)} systems clean")
print(f"  - Recovery success: {len([f for f in functional_tests if f['tests_passed'] == f['total_tests']])}/{len(affected_systems)} systems fully functional")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
